# DMOSpeech 2: Three Scheduler Revisions in One Comparative Experiment
## Development-Stage Cross-Validation & Baseline Comparison on 100 Development Speakers

**Author:** Rhenel Jhon Sajol (DMOSpeech 2 Project)  
**Evaluation Scope:** 100 Development Speakers Only (200 Cases: Clean & Noisy15)  
**Candidate Grid:** {2, 3, 4, 5, 6, 8} (1,200 Candidate Records)  
**Status:** **Development-Stage Exploratory Analysis & Pre-Run Audit**  

### Objectives & Hypotheses:
1. **Hypothesis:** Can multi-parameter early handover schedulers (Revisions 1, 2, and 3) achieve a superior quality–computation trade-off compared to simple SNR-only policies (SNR-only 2/8, 3/8, 4/8) and the original Candidate A trend ratio rule?
2. **Constraint:** Shared-data experiment on the 100 development speakers. No calibration or test speakers are accessed during threshold selection or rule formulation.
3. **Deliverables:** Unified 13-policy comparison table, paired baseline statistical comparisons, deep case audit, conditional genuine online GPU timing, and master report answering all 6 core thesis questions.

In [ ]:
# Cell 1: Environment Detection, Repository Setup, Bundle Extraction & Configuration Paths
import os
import sys
import subprocess
import hashlib
import zipfile
import shutil

# In Google Colab (/content), locate or clone repository and unpack latest revisions
IN_COLAB = ('google.colab' in sys.modules) or os.path.exists('/content')
if IN_COLAB:
    content_dir = '/content'
    repo_path = os.path.join(content_dir, 'Shonget-DMOSpeech2')
    if not os.path.exists(repo_path):
        print("[Setup] Cloning repository into /content/Shonget-DMOSpeech2 on branch early-handover-dev...")
        subprocess.run(['git', 'clone', '-b', 'early-handover-dev', 'https://github.com/Shonget-Thesis/Shonget-DMOSpeech2.git', repo_path], check=True)
    os.chdir(repo_path)
    ROOT_DIR = repo_path
    # Ensure required dependencies are installed in Google Colab environment
    try:
        import torchdiffeq
    except ImportError:
        print("[Setup] Installing required dependencies (torchdiffeq, vocos, soundfile)...")
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torchdiffeq', 'vocos', 'soundfile'], check=True)
else:
    def find_project_root():
        curr = os.getcwd()
        while curr and os.path.dirname(curr) != curr:
            if os.path.exists(os.path.join(curr, 'requirements.txt')) and (os.path.exists(os.path.join(curr, 'src')) or os.path.exists(os.path.join(curr, 'Handover Scheduling'))):
                return curr
            if os.path.exists(os.path.join(curr, 'Shonget-DMOSpeech2')):
                return os.path.abspath(os.path.join(curr, 'Shonget-DMOSpeech2'))
            curr = os.path.dirname(curr)
        return os.getcwd()
    ROOT_DIR = find_project_root()
    os.chdir(ROOT_DIR)

# Explicit mapping dictionary from archive paths / aliases to actual repository runtime destinations
TARGET_RUNTIME_MAPPING = {
    # Core runner scripts
    'src/run_three_scheduler_revisions_dev.py': [
        'colab/run_three_scheduler_revisions_dev.py',
        'Handover Scheduling/colab/run_three_scheduler_revisions_dev.py',
    ],
    'colab/run_three_scheduler_revisions_dev.py': [
        'colab/run_three_scheduler_revisions_dev.py',
        'Handover Scheduling/colab/run_three_scheduler_revisions_dev.py',
    ],
    'Handover Scheduling/colab/run_three_scheduler_revisions_dev.py': [
        'colab/run_three_scheduler_revisions_dev.py',
        'Handover Scheduling/colab/run_three_scheduler_revisions_dev.py',
    ],
    # Signals module
    'src/early_handover_signals.py': [
        'colab/early_handover_signals.py',
        'Handover Scheduling/colab/early_handover_signals.py',
    ],
    'colab/early_handover_signals.py': [
        'colab/early_handover_signals.py',
        'Handover Scheduling/colab/early_handover_signals.py',
    ],
    # Sampler module
    'src/sampler.py': [
        'Handover Scheduling/core/sampler.py',
    ],
    'Handover Scheduling/core/sampler.py': [
        'Handover Scheduling/core/sampler.py',
    ],
    # Dataset & manifest
    'data/development_expanded_grid.csv': [
        'Handover Scheduling/colab/results/early_handover_dev_gpu/development_expanded_grid.csv',
    ],
    'Handover Scheduling/colab/results/early_handover_dev_gpu/development_expanded_grid.csv': [
        'Handover Scheduling/colab/results/early_handover_dev_gpu/development_expanded_grid.csv',
    ],
    'data/development_manifest.json': [
        'Handover Scheduling/data/splits/development_manifest.json',
    ],
    'Handover Scheduling/data/splits/development_manifest.json': [
        'Handover Scheduling/data/splits/development_manifest.json',
    ],
}

# Search for uploaded verification bundle in common locations
candidate_bundle_locations = [
    '/content/three_scheduler_revisions_verification_bundle.zip',
    os.path.join(ROOT_DIR, '..', 'three_scheduler_revisions_verification_bundle.zip'),
    os.path.join(ROOT_DIR, 'three_scheduler_revisions_verification_bundle.zip'),
]

bundle_zip = next((loc for loc in candidate_bundle_locations if os.path.exists(loc)), None)
if bundle_zip:
    print(f"[Setup] Found uploaded verification bundle: {bundle_zip}. Installing into runtime folders...")
    with zipfile.ZipFile(bundle_zip, 'r') as zf:
        for member in zf.namelist():
            if member.endswith('/') or member.endswith('\\'):
                continue
            norm_member = member.replace('\\', '/')
            member_bytes = zf.read(member)
            
            targets = list(TARGET_RUNTIME_MAPPING.get(norm_member, []))
            direct_target = os.path.join(ROOT_DIR, norm_member)
            if direct_target not in [os.path.join(ROOT_DIR, t) for t in targets]:
                if any(norm_member.startswith(prefix) for prefix in ['colab/', 'Handover Scheduling/', 'data/', 'docs/', 'notebook/', 'results/']):
                    targets.append(norm_member)
            
            for rel_target in targets:
                dest_path = os.path.join(ROOT_DIR, rel_target)
                os.makedirs(os.path.dirname(dest_path), exist_ok=True)
                with open(dest_path, 'wb') as f_out:
                    f_out.write(member_bytes)
                print(f"  [Installed] {norm_member} -> {rel_target}")
    print("[Setup] All verification bundle files mapped and installed.")
else:
    print("[Setup] No uploaded bundle zip found. Using repository files directly.")

# Add exact import paths to sys.path (prioritizing colab and Handover Scheduling)
for p in [
    ROOT_DIR,
    os.path.join(ROOT_DIR, 'colab'),
    os.path.join(ROOT_DIR, 'Handover Scheduling', 'colab'),
    os.path.join(ROOT_DIR, 'Handover Scheduling', 'core'),
    os.path.join(ROOT_DIR, 'Handover Scheduling'),
    os.path.join(ROOT_DIR, 'src'),
]:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

CONFIG = {
    "experiment_name": "three_scheduler_revisions_dev",
    "manifest_path": os.path.join(ROOT_DIR, "Handover Scheduling/data/splits/development_manifest.json"),
    "grid_csv_path": os.path.join(ROOT_DIR, "Handover Scheduling/colab/results/early_handover_dev_gpu/development_expanded_grid.csv"),
    "output_dir": os.path.join(ROOT_DIR, "colab/results/three_scheduler_revisions_dev"),
    "candidate_steps": [2, 3, 4, 5, 6, 8],
    "step_nfe_map": {2: 9, 3: 11, 4: 13, 5: 15, 6: 17, 8: 21},
    "robust_risk_target": 0.05,  # 5% robust speaker failure development screen
    "n_outer_folds": 5,
    "random_seed": 42,
    "run_online_timing": True,   # Conditional sequential online GPU timing
    "timing_cases": 50,          # Number of cases to benchmark if GPU available
}

os.makedirs(CONFIG["output_dir"], exist_ok=True)
print(f"[Config Initialized] Working Directory / Root: {ROOT_DIR}")
for k, v in CONFIG.items():
    print(f"  {k:22s}: {v}")


### Cell 2: Colab Environment Logging & Checkpoints Verification
Logs allocated GPU hardware, available VRAM, host RAM, Python/PyTorch/CUDA versions, precision, runner SHA256, and verifies required checkpoints and dataset audio.

In [ ]:
# Cell 2: Hardware Logging & Checkpoints / Module Verification
import platform
import psutil

print("=== Host Hardware & Software Environment ===")
print(f"OS Platform    : {platform.platform()}")
print(f"Python Version : {platform.python_version()}")
print(f"Host CPU Cores : {os.cpu_count()}")
print(f"Host RAM (GB)  : {psutil.virtual_memory().total / (1024**3):.2f} GB")

try:
    import torch
    print(f"PyTorch Version: {torch.__version__}")
    print(f"CUDA Available : {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU Model      : {torch.cuda.get_device_name(0)}")
        print(f"GPU VRAM (GB)  : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
        print(f"cuDNN Version  : {torch.backends.cudnn.version()}")
except ImportError:
    print("PyTorch not installed in base environment.")

try:
    import torchdiffeq
    print("torchdiffeq     : FOUND")
except ImportError:
    print("torchdiffeq     : MISSING")

# Source module provenance and SHA256 hashes
def get_sha256(path):
    if not os.path.exists(path):
        return 'missing'
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while chunk := f.read(65536):
            h.update(chunk)
    return h.hexdigest()

# Verify and print actual imported module paths and hashes
import run_three_scheduler_revisions_dev
print("\n=== Imported Module Verification ===")
runner_file = getattr(run_three_scheduler_revisions_dev, '__file__', 'unknown')
print(f"Imported Runner : {runner_file}")
print(f"Runner SHA256   : {get_sha256(runner_file)}")

try:
    import sampler
    sampler_file = getattr(sampler, '__file__', 'unknown')
    print(f"Imported Sampler: {sampler_file}")
    print(f"Sampler SHA256  : {get_sha256(sampler_file)}")
except ImportError:
    sampler_file = os.path.join(ROOT_DIR, 'Handover Scheduling', 'core', 'sampler.py')
    print(f"Sampler Module  : {sampler_file} (SHA256: {get_sha256(sampler_file)})")

# Checkpoints & Dataset Verification
print("\n=== Data & Checkpoints Verification ===")
ckpt_model = os.path.join(ROOT_DIR, 'ckpts', 'model_85000.pt')
ckpt_dur = os.path.join(ROOT_DIR, 'ckpts', 'model_1500.pt')
print(f"Voice Checkpoint : {'FOUND' if os.path.exists(ckpt_model) else 'MISSING'} ({ckpt_model})")
print(f"Dur Checkpoint   : {'FOUND' if os.path.exists(ckpt_dur) else 'MISSING'} ({ckpt_dur})")
print(f"Manifest Exists  : {'FOUND' if os.path.exists(CONFIG['manifest_path']) else 'MISSING'}")
print(f"Grid CSV Exists  : {'FOUND' if os.path.exists(CONFIG['grid_csv_path']) else 'MISSING'}")
if os.path.exists(CONFIG['grid_csv_path']):
    import pandas as pd
    df_chk = pd.read_csv(CONFIG['grid_csv_path'])
    print(f"Grid Records     : {len(df_chk)} rows (expected 1,200) | SHA256: {get_sha256(CONFIG['grid_csv_path'])}")
    assert len(df_chk) == 1200, f"Grid CSV must have exactly 1,200 rows! Found {len(df_chk)}"
print("[Setup] Environment & module imports verified successfully.")


### Cell 3: Required Inputs Loading & 7-Point Mandatory Verification
Validates the 7 formal requirements from Section 3 before any analysis:
1. Exactly 100 dev speaker IDs, 2 conditions per speaker, 6 candidate indices per case.
2. Exactly 1,200 unique records with zero missing quality or decision inputs.
3. Reference values match corresponding index-8 records identically.
4. Failure flags reproduce the original reference-relative definition.
5. NFE equals $2(s+1)+3$ for all candidate indices.
6. Compatibility and provenance verification.
7. Online signal causality check (signals at step $i$ only use information available at step $i$).
**Safety Guarantee:** Execution will immediately stop with an exception if any check fails.

In [ ]:
# Cell 3: Load Inputs & Execute 7-Point Verification
import json
import pandas as pd
import numpy as np
from run_three_scheduler_revisions_dev import (
    validate_inputs,
    build_case_dictionary,
    verify_executable_causality,
)

with open(CONFIG['manifest_path'], 'r', encoding='utf-8') as f:
    manifest_cases = json.load(f)
df_grid = pd.read_csv(CONFIG['grid_csv_path'])

print(f"Loaded Manifest: {len(manifest_cases)} cases")
print(f"Loaded Grid    : {len(df_grid)} candidate evaluations")

# 1. Build Case Dictionary & Verify Causality
case_dict = build_case_dictionary(df_grid)
causality_report = verify_executable_causality(case_dict, config=CONFIG)

# 2. Run 7-Point Verification (raises RuntimeError if validation fails)
val_report = validate_inputs(df_grid, manifest_cases, CONFIG['output_dir'], config=CONFIG, causality_report=causality_report)
print(f"=== 7-Point Validation Passed: {val_report['validation_passed']} ===")
for check_name, check_data in val_report['checks'].items():
    status = 'PASS' if check_data.get('passed', True) else 'FAIL'
    print(f"  [{status}] {check_name}")


### Cell 4: Baseline Policies Reconstruction & Baseline Reconciliation
Reconstructs the original frozen baselines and reconciles with historical development results:
- **Fixed Index 8 Reference:** Full teacher completion (21 NFE, 0 failures).
- **SNR-Only 4/8:** Original proposed early-exit baseline (17 NFE, 3 failures).
- **SNR-Only 3/8:** Stronger intermediate fixed early exit (16 NFE, 3 failures).
- **SNR-Only 2/8:** Cheapest fixed early exit (15 NFE, 7 failures).
- **Original SNR + Drift (Reconciled):** Handover at earliest $s \in \{4, 5, 6\}$ with $d_s \le 0.50$, else 8 (**18.05 NFE, 1 failure**, matching `expanded_policy_comparison.csv` and `early_handover_case_audit.csv`).
- **Candidate A Replay ($b=1.0$):** Handover at earliest $s \in \{2, 3, 4, 5, 6\}$ with $r_s \le 1.0$, else 8 (**16.25 NFE, 2 failures**).

In [ ]:
# Cell 4: Build Fast Case Dictionary & Evaluate Baselines
from run_three_scheduler_revisions_dev import (
    build_case_dictionary,
    policy_fixed,
    policy_snr_only,
    policy_original_snr_drift,
    policy_candidate_a,
    evaluate_policy_on_speakers,
)

case_dict = build_case_dictionary(df_grid)
speakers = sorted(list(set(k[0] for k in case_dict.keys())))

baseline_checks = [
    ("Fixed_Index_8", policy_fixed(8)),
    ("SNR_Only_4_8", policy_snr_only(4)),
    ("SNR_Only_3_8", policy_snr_only(3)),
    ("SNR_Only_2_8", policy_snr_only(2)),
    ("Original_SNR_Plus_Drift", policy_original_snr_drift(0.50)),
    ("Candidate_A_Replay_1.0", policy_candidate_a(1.0)),
]

baseline_summary = []
for name, p_fn in baseline_checks:
    res = evaluate_policy_on_speakers(p_fn, speakers, case_dict)
    baseline_summary.append({
        "policy": name,
        "mean_nfe": res["mean_nfe"],
        "nfe_reduction_%": res["nfe_reduction_vs_step8_%"],
        "robust_failures": res["robust_failures"],
        "robust_risk_%": res["robust_failure_rate_%"],
        "clean_wer": res["clean_mean_wer"],
        "clean_sim": res["clean_mean_sim"],
        "exit_distribution": str(res["exit_counts"]),
    })

df_base_audit = pd.DataFrame(baseline_summary)
display(df_base_audit)


### Cell 5: Implement Three Distinct Scheduler Revisions & Executable Causality Audit
All revisions retain pre-generation SNR gating (prompt SNR < 20 dB $\to$ Index 8).
For eligible clean speech (SNR $\ge 20$ dB), candidates examine indices $\{2, 3, 4, 5, 6\}$:
1. **Revision 1 (Trend + Absolute Drift):** Stop when $d_i \le a \text{ AND } r_i \le b$.
2. **Revision 2 (Trend + Conditioning Disagreement):** Stop when $r_i \le b \text{ AND } c_i \le c$ (and opposite direction check $c_i \ge c'$).
3. **Revision 3 (Index-Specific Trend Groups):** Stop when $r_i \le b_i$ with groups: $b_2$ for step 2, $b_3$ for step 3, $b_{\text{later}}$ for steps 4, 5, 6.

**Formal Causality Verification:** Uses `CausalOnlineStepView` to actively block access to future steps ($> i$), future drift columns ($d_{i+1}$), and offline quality metrics (`wer`, `is_failure`).

In [ ]:
# Cell 5: Policy Implementations & Executable Online Causality Verification
from run_three_scheduler_revisions_dev import (
    policy_revision_1,
    policy_revision_2,
    policy_revision_3,
    verify_executable_causality,
)

# Run formal causality test suite (negative unit tests + 100-speaker positive integration)
causality_report = verify_executable_causality(case_dict, config=CONFIG)
assert causality_report['negative_tests_passed'], 'Negative causality unit tests failed!'
assert causality_report['positive_integration_passed'], 'Positive causality integration test failed!'

print("[PASS] Executable online causality verified across all 100 development speakers!")
print(f"  - Blocked future step access   : {causality_report['details']['blocked_future_step']}")
print(f"  - Blocked future drift columns : {causality_report['details']['blocked_future_column']}")
print(f"  - Blocked offline quality label: {causality_report['details']['blocked_quality_feature']}")
print(f"  - Evaluated speakers           : {causality_report['checked_speakers']} (0 violations detected)")


### Cell 6: Verify Telemetry Completeness Across Shared Candidate Grid
Verifies that all 1,200 grid candidate rows have complete evaluations and signals ($d_2 \dots d_6$, $q_A$, $q_C$, WER, SIM, UTMOS) to avoid redundant synthesis.

In [ ]:
# Cell 6: Candidate Grid Telemetry Audit
signals = ["d_2", "d_3", "d_4", "d_5", "d_6", "q_A", "q_C", "wer", "similarity", "utmos"]
clean_df = df_grid[df_grid["condition"] == "clean"]

missing_telemetry = {}
for sig in signals:
    missing_telemetry[sig] = int(clean_df[sig].isna().sum())

print("Missing telemetry count in 600 clean candidate records:")
for sig, cnt in missing_telemetry.items():
    print(f"  {sig:12s}: {cnt} missing")

assert all(cnt == 0 for cnt in missing_telemetry.values()), "Error: missing telemetry in development grid!"
print("[PASS] All required candidate signals exist; zero audio re-synthesis needed for quality grid.")


### Cell 7: Predefined Parameter Search Grids & 5 Speaker-Disjoint Outer Folds
Sets up deterministic 5-fold cross-validation on the 100 development speakers (`KFold(n_splits=5, shuffle=True, random_state=42)`).
Both clean and noisy15 conditions of a speaker remain strictly in the same fold.

In [ ]:
# Cell 7: Setup Folds and Predefined Search Spaces
from sklearn.model_selection import KFold

kf = KFold(n_splits=CONFIG['n_outer_folds'], shuffle=True, random_state=CONFIG['random_seed'])
spk_arr = np.array(speakers)

print(f"Created {CONFIG['n_outer_folds']} speaker-disjoint folds across {len(speakers)} speakers.")
for fold_idx, (tr_idx, val_idx) in enumerate(kf.split(spk_arr)):
    print(f"  Fold {fold_idx}: Train {len(tr_idx)} speakers | Validation {len(val_idx)} speakers")


### Cell 8: Training-Fold Parameter Search & Held-Out Outer Validation
Executes the full cross-validation workflow:
1. Evaluates all search grid combinations on the 80 training speakers per fold.
2. Screens candidates for robust risk $\le 5.0\%$ and selects setting minimizing mean NFE.
3. Freezes that fold's selected parameter rule and evaluates on the 20 held-out validation speakers once.
4. Exports `threshold_search_log.csv`, `revision_fold_results.csv`, and `out_of_fold_case_decisions.csv`.

In [ ]:
# Cell 8: Run 5-Fold Cross-Validation Workflow
from run_three_scheduler_revisions_dev import run_revisions_cross_validation

df_spk_folds, df_search, df_folds, df_oof = run_revisions_cross_validation(
    case_dict=case_dict,
    output_dir=CONFIG['output_dir'],
    config=CONFIG,
)

print("=== Out-of-Fold Cross-Validation Results Summary ===")
oof_summary = df_folds.groupby('candidate_formula')[['val_robust_risk_%', 'val_mean_nfe']].mean()
display(oof_summary)

print("\n=== Selected Parameters Per Fold ===")
display(df_folds[['fold', 'candidate_formula', 'selected_params', 'val_robust_fails', 'val_mean_nfe', 'val_step2_exits', 'val_step3_exits', 'val_step4_exits']])


### Cell 9: Unified 13-Policy Comparison Table & Paired Baseline Statistical Tests
Evaluates all 13 policies on the same 200 cases (100 clean, 100 noisy15):
- 9 Deployable policies (Fixed 8 reference, 4 fixed baselines, Candidate A OOF, Revision 1 OOF, Revision 2 OOF, Revision 3 OOF).
- Computes speaker-level paired differences across 19 comparisons against Fixed 8, SNR-only 4/8, SNR-only 3/8, Original SNR+drift, and Candidate A.
- McNemar exact two-tailed tests on discordant failure pairs.
- 95% speaker cluster bootstrap confidence intervals on NFE difference.
- Wilcoxon signed-rank tests.

In [ ]:
# Cell 9: Unified Table & Paired Baseline Comparisons
from run_three_scheduler_revisions_dev import run_all_policy_comparison

df_all_pol, df_paired, df_audit = run_all_policy_comparison(
    case_dict=case_dict,
    df_oof=df_oof,
    output_dir=CONFIG['output_dir'],
    config=CONFIG,
)

print("=== Unified 13-Policy Comparison Table ===")
display(df_all_pol[['policy', 'status', 'mean_nfe', 'nfe_reduction_%', 'robust_failures', 'robust_risk_%', 'clean_wer', 'clean_sim', 'step2_exits', 'step3_exits', 'step4_exits', 'step8_exits']])

print("\n=== Paired Statistical Comparisons (19 Rows) ===")
display(df_paired[['revision_policy', 'baseline_policy', 'rescued_failures', 'harmed_failures', 'net_avoided_failures', 'mcnemar_exact_p_value', 'mean_nfe_difference', 'nfe_diff_95%_ci_lower', 'nfe_diff_95%_ci_upper', 'wilcoxon_nfe_p_value']])


### Cell 10: Deep-Dive Case Audit (Rescued, Harmed & Unnecessary Waiting)
Examines individual speaker cases to understand where adaptive rules succeed or fail:
- **Rescued Cases:** Cases that failed under a fixed baseline but passed under the adaptive revision.
- **Harmed Cases:** Cases that passed under a fixed baseline but failed under the adaptive revision.
- **Unnecessary Delays:** Cases where early exits (Step 2 or 3) were acoustically safe, but the policy waited until later steps.

In [ ]:
# Cell 10: Case-Level Failure & Waiting Audit
audit_file = os.path.join(CONFIG['output_dir'], 'failure_and_waiting_case_audit.csv')
df_case_audit = pd.read_csv(audit_file)

# 1. Unnecessary waiting cases
unnecessary_wait_cases = df_case_audit[df_case_audit.get('revision_1_unnecessary_waiting', False) == True]
print(f"Cases where Step 3 passed but Revision 1 waited unnecessarily: {len(unnecessary_wait_cases)} / 100 speakers")
if len(unnecessary_wait_cases) > 0:
    wait_cols = ['speaker_id', 'step3_clean_passed', 'Revision_1_OOF_clean_exit', 'SNR_Only_3_8_clean_exit', 'Fixed_Index_8_clean_exit']
    display(unnecessary_wait_cases[[c for c in wait_cols if c in unnecessary_wait_cases.columns]].head(10))

# 2. Robust failure cases
r1_fail_col = 'Revision_1_OOF_robust_failure' if 'Revision_1_OOF_robust_failure' in df_case_audit.columns else 'Revision_1_OOF_robust_fail'
s3_fail_col = 'SNR_Only_3_8_robust_failure' if 'SNR_Only_3_8_robust_failure' in df_case_audit.columns else 'SNR_Only_3_8_robust_fail'
rob_fail_cases = df_case_audit[(df_case_audit[r1_fail_col] == True) | (df_case_audit[s3_fail_col] == True)]
print("\nSpeaker cases with robust failure in either Revision 1 or SNR-only 3/8:")
fail_cols = ['speaker_id', 'Revision_1_OOF_clean_exit', r1_fail_col, 'SNR_Only_3_8_clean_exit', s3_fail_col]
display(rob_fail_cases[[c for c in fail_cols if c in rob_fail_cases.columns]])


### Cell 11: Conditional Genuine Online GPU Timing Benchmark Pipeline
Executes the genuine utterance synthesis pipeline on a single shared GPU instance:
1. **Pre-Benchmark Smoke Test:** Evaluates 1 clean case and 1 noisy case across all 6 timed policies, verifying prompt SNR estimation, candidate index restriction (stops in {2,3,4,5,6}, fallback 8), model call counts via actual instrumentation, student completion, and finite waveform generation.
2. **Overhead-Free Baselines:** Fixed 8 and SNR-only run pure inference without tangent drift, trend ratio, or .item() GPU synchronization.
3. **Synchronized Timer Boundary:** CUDA synchronized before teacher Euler integration and after Vocos vocoder decode. Input preparation overhead measured separately.
4. **Balanced Execution:** Cycles policy order across cases to prevent thermal/caching bias.
5. **Audited Deliverables:** Exports `case_level_timing_audit.csv` (per-case latency, calls, and RTF) and `online_policy_timing.csv` (aggregated metrics vs Fixed 8).

In [ ]:
# Cell 11: Conditional Genuine Online GPU Latency Benchmarking
import torch
from run_three_scheduler_revisions_dev import run_conditional_gpu_timing

df_timing = None
if torch.cuda.is_available() and CONFIG.get('run_online_timing', True):
    print("[Timing] CUDA GPU detected. Running smoke test then genuine online GPU latency benchmarking...")
    df_timing = run_conditional_gpu_timing(
        manifest_cases=manifest_cases,
        output_dir=CONFIG['output_dir'],
        device="cuda",
        run_online=True,
        smoke_test_only=False,
        config=CONFIG,
        case_dict=case_dict,
    )
    if df_timing is not None:
        display(df_timing)
else:
    print("[Timing] CUDA unavailable or timing skipped. Latency marked as 'not measured' per protocol.")
    df_timing = run_conditional_gpu_timing(
        manifest_cases=manifest_cases,
        output_dir=CONFIG['output_dir'],
        device="cpu",
        run_online=False,
        smoke_test_only=False,
        config=CONFIG,
        case_dict=case_dict,
    )


### Cell 12: Export Master Report, Experiment Metadata & Downloadable Archive
Compiles `THREE_SCHEDULER_REVISIONS_DEVELOPMENT_REPORT.md` answering all 6 core questions, logs experiment metadata, and packages all 14 artifacts into `three_scheduler_revisions_dev_artifacts.tar.gz` for download.

In [ ]:
# Cell 12: Compile Master Report & Download Archive
import datetime
import tarfile
from run_three_scheduler_revisions_dev import (
    generate_master_development_report,
    safe_json_serialize,
    compute_file_sha256,
    get_git_commit_hash,
)

# Generate Report
generate_master_development_report(
    df_all_pol=df_all_pol,
    df_paired=df_paired,
    df_folds=df_folds,
    df_timing=df_timing,
    val_report=val_report,
    output_dir=CONFIG['output_dir'],
)

# Export Metadata
meta = {
    "timestamp": datetime.datetime.now().isoformat(),
    "git_commit": get_git_commit_hash(),
    "platform": platform.platform(),
    "python_version": platform.python_version(),
    "torch_version": torch.__version__ if 'torch' in sys.modules else 'not_installed',
    "cuda_available": torch.cuda.is_available() if 'torch' in sys.modules else False,
    "n_development_speakers": 100,
    "n_cases": 200,
    "candidate_steps": CONFIG['candidate_steps'],
    "n_grid_records": len(df_grid),
    "sha256_hashes": {
        "grid_csv": compute_file_sha256(CONFIG['grid_csv_path']),
        "manifest": compute_file_sha256(CONFIG['manifest_path']),
    }
}
with open(os.path.join(CONFIG['output_dir'], 'experiment_metadata.json'), 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, default=safe_json_serialize)

# Package Archive
archive_path = os.path.join(CONFIG['output_dir'], 'three_scheduler_revisions_dev_artifacts.tar.gz')
with tarfile.open(archive_path, 'w:gz') as tar:
    for fname in os.listdir(CONFIG['output_dir']):
        if fname != 'three_scheduler_revisions_dev_artifacts.tar.gz':
            tar.add(os.path.join(CONFIG['output_dir'], fname), arcname=fname)

print(f"[Archive Ready] {archive_path} ({os.path.getsize(archive_path) / 1024:.1f} KB)")

# Trigger Google Colab browser download if running in Colab
try:
    from google.colab import files
    files.download(archive_path)
    print("[Download] Archive download initiated in browser.")
except Exception as e:
    print(f"[Download Note] Running outside Google Colab. Artifact archive saved at {archive_path}")
